# One channel against the HFSS beam, in az/el and in time

One transmitter comb tooth through the elevation-sweep raster (07-17
20:26–21:28 UTC), plotted on the corrected pointing coordinates
(`az_deg`, `el_deg` from `curation/pointing_table.parquet`). It is compared with
the HFSS polarised beam sampled at the same coordinates, with the residual.

The signal is the tooth minus the mean of the gap channels ±4 away. The model
is `gain × |E_beam* · e_tx|²`. The only fitted parameter is the gain: the
background is removed by the gap subtraction, not fitted, with the beam pointed at the
triangulated transmitter. The mount rolls about the boresight
(`topo = Rx(E)·Rz(A)·body`, as in `eigsep_data.beam_mapping.tx_model`), with the
zero-points and transmitter polarisation taken from the per-frequency fits in
`beam_explorer.ipynb`. Only the gain and offset are fitted here.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import healpy as hp
import pyarrow.parquet as pq

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)
os.environ.setdefault('JAX_PLATFORMS', 'cpu')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'

import eigsep_data
from eigsep_data import AntennaResolutionPolicy, MetadataIndex
from eigsep_data.beam_sim import read_beam
eigsep_data.set_campaign_root(CAMPAIGN)

RASTER_UTC = ('2026-07-17 20:26:00', '2026-07-17 21:28:40')
FREQ_MHZ = 156.25            # an HFSS slice; every HFSS slice sits on an arm-1 comb tooth
GAP_OFFSET = 4

# Geometry and polarisation from beam_explorer.ipynb (branch B, roll mount).
AZ_SIGN, EL_SIGN = -1.0, -1.0
AZ0, EL0 = 177.0, 0.0        # deg
ALPHA = 45.0                 # transmitter polarisation, deg ccw from East

In [ ]:
t0, t1 = (pd.Timestamp(s, tz='UTC').timestamp() for s in RASTER_UTC)

# --- corrected pointing, joined to the correlator rows on time -------------
pointing = pq.read_table(CAMPAIGN / 'curation/pointing_table.parquet').to_pandas()
pointing = pointing[(pointing.t_utc_s >= t0 - 60) & (pointing.t_utc_s <= t1 + 60)]
policy = AntennaResolutionPolicy.load(CAMPAIGN / 'curation/antenna_resolution.json')
bundle = MetadataIndex(CAMPAIGN / 'data').select(time=(t0, t1)).load_bundle(
    antenna='box-air', missing='skip', resolution_policy=policy)
meta = bundle.meta.reset_index(drop=True).assign(t=lambda x: x.time_best.astype(float))
j = pd.merge_asof(meta.sort_values('t').reset_index(names='bundle_row'),
                  pointing.sort_values('t_utc_s')[['t_utc_s', 'az_deg', 'el_deg', 'quality']],
                  left_on='t', right_on='t_utc_s', direction='nearest', tolerance=0.35)
j = j[(j.quality == 'ok') & (j.rfswitch == 'RFANT')].reset_index(drop=True)

def reversal_glitches(el_deg, tol=15.0):
    """Isolated solved-elevation faults: a sample is good if at least 3 of its
    4 neighbours (+/-1, +/-2) lie within `tol` deg of it across the +/-180
    wrap. The sweeps move ~3 deg per sample, so real data always passes.
    At reversals the IMU readout drops single samples to ~0 or -60 deg while
    the motor is pegged at -180, and those have at most one close neighbour.
    """
    el = np.asarray(el_deg, float)
    close = np.zeros(el.size, int)
    for k in (-2, -1, 1, 2):
        d = np.abs((el - np.roll(el, k) + 180.0) % 360.0 - 180.0)
        edge = slice(0, -k) if k < 0 else slice(-k, None)   # neighbours that wrapped around the array
        d[edge] = 0.0
        close += d <= tol
    return close < 3


glitch = reversal_glitches(j.el_deg.to_numpy())
print('dropped %d elevation glitches' % glitch.sum())
j = j[~glitch].reset_index(drop=True)

T = j.t.to_numpy()
AZ = j.az_deg.to_numpy()
EL = j.el_deg.to_numpy()

# --- the channel -------------------------------------------------------------
BEAM_CART, _, HFSS_FREQS = read_beam()
s = int(np.argmin(np.abs(HFSS_FREQS - FREQ_MHZ)))
ch = int(round(HFSS_FREQS[s] / (250.0 / 1024)))
col = {int(round(f / (250.0 / 1024))): i for i, f in enumerate(bundle.freqs_mhz)}
rows = j.bundle_row.to_numpy()
tooth = bundle.data[rows, col[ch]].astype(float)
gap = 0.5 * (bundle.data[rows, col[ch - GAP_OFFSET]] + bundle.data[rows, col[ch + GAP_OFFSET]])
DATA = tooth - gap
# Whole-band dropouts (e.g. 21:18:00-21:18:35) take the gap down with the tooth.
ok = (gap / np.median(gap) > 1 / 1.5) & (gap / np.median(gap) < 1.5) & np.isfinite(DATA)
print('channel %d = %.4f MHz; %d samples, %d masked as dropouts'
      % (ch, HFSS_FREQS[s], len(DATA), (~ok).sum()))

In [ ]:
# --- HFSS model at the same coordinates -------------------------------------
ant = np.asarray(json.loads((CAMPAIGN / 'curation/horizon_profiles.json').read_text())['antenna_enu_m'])
tx = np.asarray(json.loads((CAMPAIGN / 'curation/transmitter_position.json').read_text())['best_estimate_enu_m'])
tx_dir = (tx - ant) / np.linalg.norm(tx - ant)


def rotation(az_deg, el_deg):
    # topo = Rx(E) @ Rz(A) @ body: azimuth rolls about the boresight, elevation tips.
    A, E = np.deg2rad(az_deg), np.deg2rad(el_deg)
    cA, sA, cE, sE = np.cos(A), np.sin(A), np.cos(E), np.sin(E)
    z, o = np.zeros_like(cA), np.ones_like(cA)
    Rz = np.stack([np.stack([cA, -sA, z], -1), np.stack([sA, cA, z], -1), np.stack([z, z, o], -1)], -2)
    Rx = np.stack([np.stack([o, z, z], -1), np.stack([z, cE, -sE], -1), np.stack([z, sE, cE], -1)], -2)
    return Rx @ Rz


R = rotation(AZ_SIGN * AZ + AZ0, EL_SIGN * EL + EL0)
b = np.einsum('nji,j->ni', R, tx_dir)                       # transmitter direction, body frame
th, ph = np.arccos(np.clip(b[:, 2], -1, 1)), np.mod(np.arctan2(b[:, 1], b[:, 0]), 2 * np.pi)
w = BEAM_CART[s][:, hp.ang2pix(hp.npix2nside(BEAM_CART.shape[2]), th, ph)].T
a = np.deg2rad(ALPHA)
e = np.einsum('nji,j->ni', R, np.array([-np.sin(a), np.cos(a), 0.0]))
e -= np.sum(e * b, 1, keepdims=True) * b
w -= np.sum(w * b, 1, keepdims=True) * b
P = np.abs(np.sum(np.conj(w) * e, 1)) ** 2

# One free parameter: the gain. The background is removed by the gap
# subtraction itself (DATA = tooth - gap); nothing further is fitted for it.
# The gap channel is 1 MHz away with the tones bin-centred and leakage-free, so
# it measures the background under the tooth directly. The published
# v3-beta.1 DPSS model was compared against it on the raster and is
# indistinguishable (mean R^2 0.744 vs 0.745, <=0.003 at every frequency),
# while covering only 13 of the 30 raster files, so the gap is used.
gain = float(np.sum(P[ok] * DATA[ok]) / np.sum(P[ok] ** 2))
MODEL = gain * P
RESID = DATA - MODEL
print('gain %.3g;  R^2 %.3f' % (gain, 1 - RESID[ok].var() / DATA[ok].var()))

## Figure 1 — data, model and residual on the corrected az/el

Each vertical stripe is one elevation sweep at a (nearly) fixed azimuth. The
single horizontal row at el ≈ −1° is the azimuth slew at the start of the raster
(20:26:00–20:26:30). There the platform swings 166° in azimuth at constant
elevation, so it is a genuine azimuth cut, not a glitch.


In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False})
az_plot = (AZ + 180.0) % 360.0 - 180.0          # the raster crosses 0/360; keep it contiguous

fig, axes = plt.subplots(1, 3, figsize=(19, 7), sharex=True, sharey=True,
                         constrained_layout=True)
lo, hi = np.percentile(DATA[ok & (DATA > 0)], [1, 99.5])
norm = LogNorm(vmin=lo, vmax=hi)
rlim = np.percentile(np.abs(RESID[ok]), 98)
panels = [('Data (tooth − gap)', DATA, dict(cmap='viridis', norm=norm), 'counts'),
          ('HFSS model, same coordinates', MODEL, dict(cmap='viridis', norm=norm), 'counts'),
          ('Residual (data − model)', RESID, dict(cmap='RdBu_r', vmin=-rlim, vmax=rlim), 'counts')]
for ax, (title, val, kw, label) in zip(axes, panels):
    sc = ax.scatter(az_plot[ok], EL[ok], c=val[ok], s=5, **kw)
    ax.set_title(title)
    ax.set_xlabel('Azimuth [deg]')
    fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.1, shrink=0.85, label=label)
axes[0].set_ylabel('Elevation [deg]')
fig.suptitle('%.3f MHz, raster %s–%s UTC' % (HFSS_FREQS[s], RASTER_UTC[0][-8:], RASTER_UTC[1][-8:]),
             fontsize=17, y=1.04)
plt.show()

## Figure 2 — the same three against time

In [ ]:
time = pd.to_datetime(T, unit='s', utc=True)
fig, ax = plt.subplots(figsize=(16, 6.5), constrained_layout=True)
ax.plot(time[ok], DATA[ok], '.', ms=2, color='0.25', label='data')
ax.plot(time[ok], MODEL[ok], '.', ms=2, color='tab:red', label='HFSS model')
ax.plot(time[ok], RESID[ok], '.', ms=2, color='tab:blue', label='residual')
ax.axhline(0, color='0.6', lw=1)
ax.set_ylabel('Counts')
ax.set_xlabel('Time [UTC]')
ax.set_title('%.3f MHz through the raster' % HFSS_FREQS[s])
ax.legend(loc='upper right', markerscale=6)
plt.show()

In [ ]:
np.sqrt(np.mean(np.abs(RESID[ok])**2)) / np.sqrt(np.mean(np.abs(DATA[ok])**2))